# 5. Data Cleaning

### 5.1 Imports

In [1]:
import pandas as pd
import numpy as np
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))  # one level up
# Add src folder to path
sys.path.append(os.path.join(project_root, "src"))
from data_loader import load_raw_data
# Set pandas display options to show all columns
pd.set_option('display.max_columns', None)

### 5.2 Load Data

In [2]:

# ─────────────────────────────────────────────
# LOAD YOUR DATA (update paths as needed)
# ─────────────────────────────────────────────
bookings_df, customers_df, drivers_df, location_demand_df, time_features_df = load_raw_data()


### 5.3 Bookings Cleaning

#### 5.3.1 Fix Date and Time Types

In [3]:
# ════════════════════════════════════════════════════════
# 5.3 BOOKINGS CLEANING
# ════════════════════════════════════════════════════════

# ── 5.3.1 Fix Date and Time Types ───────────────────────
# Combine booking_date + booking_time into single datetime column
bookings_df["booking_datetime"] = pd.to_datetime(
    bookings_df["booking_date"].astype(str) + " " + bookings_df["booking_time"]
)
# Drop redundant separate columns
bookings_df.drop(columns=["booking_date", "booking_time"], inplace=True)

print("booking_datetime dtype:", bookings_df["booking_datetime"].dtype)
# Expected: datetime64[ns] 


booking_datetime dtype: datetime64[ns]


#### 5.3.2 Handle Missing Values

In [4]:
# ── 5.3.2 Handle Missing Values ─────────────────────────

# --- actual_ride_time_min ---
# Missing values are fully dependent on booking_status
# Completed rides → already have actual time (0 nulls)
# Cancelled + Incomplete → ride never finished → keep NaN
# Add flag column to capture missingness as a feature
bookings_df["actual_time_available"] = (
    bookings_df["actual_ride_time_min"].notna().astype(int)
)
print(bookings_df["actual_time_available"].value_counts())
# 1 → 68346 (Completed)
# 0 → 31654 (Cancelled + Incomplete)
# NaN stays as is — will be imputed AFTER train/test split 

# --- incomplete_ride_reason ---
# NaN means ride was Completed or Cancelled — not truly missing
# Verify nulls belong only to Completed + Cancelled
print(bookings_df.groupby("booking_status")["incomplete_ride_reason"].apply(
    lambda x: x.isna().sum()
))
# Fill with "Not Applicable"
bookings_df["incomplete_ride_reason"] = (
    bookings_df["incomplete_ride_reason"].fillna("Not Applicable")
)
print(bookings_df["incomplete_ride_reason"].value_counts())
# Not Applicable    91630 


actual_time_available
1    68346
0    31654
Name: count, dtype: int64
booking_status
Cancelled     23284
Completed     68346
Incomplete        0
Name: incomplete_ride_reason, dtype: int64
incomplete_ride_reason
Not Applicable      91630
Driver Delay         4728
Vehicle Issue        1265
App Issue            1221
Customer No-show     1156
Name: count, dtype: int64


#### 5.3.3 Standardise Categoricals

In [5]:
# ── 5.3.3 Standardise Categoricals ──────────────────────
for col in ["city", "vehicle_type", "traffic_level",
            "weather_condition", "booking_status",
            "incomplete_ride_reason"]:
    bookings_df[col] = bookings_df[col].str.strip().str.title()

print(bookings_df["booking_status"].value_counts())
# Completed    68346
# Cancelled    23284
# Incomplete    8370 


booking_status
Completed     68346
Cancelled     23284
Incomplete     8370
Name: count, dtype: int64


#### 5.3.4 Remove Duplicates

In [6]:
# ── 5.3.4 Remove Duplicates ──────────────────────────────
print(f"Duplicates before: {bookings_df.duplicated().sum()}")
bookings_df.drop_duplicates(inplace=True)
print(f"Shape after dedup: {bookings_df.shape}")
# Expected: (100000, 25) 

Duplicates before: 0
Shape after dedup: (100000, 22)


#### 5.3.5 Booking Value Validation

In [7]:
# ── 5.3.5 Booking Value Validation ───────────────────────
bookings_df["computed_value"] = (
    bookings_df["base_fare"] * bookings_df["surge_multiplier"]
).round(2)
mismatch = ~np.isclose(
    bookings_df["booking_value"], bookings_df["computed_value"], atol=1.0
)
print(f"Booking value mismatches: {mismatch.sum()}")
# ~90739 → booking_value includes additional business rules
# booking_value is treated as ground truth 
# Drop computed helper column
bookings_df.drop(columns=["computed_value"], inplace=True)

Booking value mismatches: 90739


#### 5.3.6 Same Pickup and Drop Location Flag

In [8]:
# ── 5.1.6 Same Pickup and Drop Location Flag ─────────────
bookings_df["same_loc_flag"] = (
    bookings_df["pickup_location"] == bookings_df["drop_location"]
).astype(int)
print(f"Same pickup/drop rows: {bookings_df['same_loc_flag'].sum()}")
# ~1979 rows flagged

Same pickup/drop rows: 1979


#### 5.3.7 Final Bookings Check

In [9]:
# ── 5.3.7 Final Bookings Check ───────────────────────────
print("\nBookings nulls:\n", bookings_df.isnull().sum()[bookings_df.isnull().sum() > 0])
print("Bookings shape:", bookings_df.shape)
# Only actual_ride_time_min should have nulls (31654) — intentional 


Bookings nulls:
 actual_ride_time_min    31654
dtype: int64
Bookings shape: (100000, 23)


### 5.4 Customers Cleaning

#### 5.4.1 Standardise Categoricals

In [10]:

# ════════════════════════════════════════════════════════
# 5.4 CUSTOMERS CLEANING
# ════════════════════════════════════════════════════════

# ── 5.4.1 Standardise Categoricals ──────────────────────
for col in ["customer_gender", "customer_city", "preferred_vehicle_type"]:
    customers_df[col] = customers_df[col].str.strip().str.title()

#### 5.4.2 Validate Customer Age Range

In [11]:

# ── 5.4.2 Validate Customer Age Range (18–64) ────────────
age_outliers = customers_df[
    (customers_df["customer_age"] < 18) |
    (customers_df["customer_age"] > 80)
]
print(f"Customer age outliers: {len(age_outliers)}")
# Expected: 0 

Customer age outliers: 0


#### 5.4.3 Validate Cancellation Rate

In [12]:

# ── 5.4.3 Validate Cancellation Rate ────────────────────
customers_df["computed_cancel_rate"] = (
    customers_df["cancelled_rides"] / customers_df["total_bookings"]
).round(6)
rate_mismatch = customers_df[~np.isclose(
    customers_df["cancellation_rate"],
    customers_df["computed_cancel_rate"], atol=0.001
)]
print(f"Cancellation rate mismatches: {len(rate_mismatch)}")
# Expected: 0 
customers_df.drop(columns=["computed_cancel_rate"], inplace=True)

Cancellation rate mismatches: 0


#### 5.4.4 Validate Total Bookings

In [13]:
# ── 5.4.4 Validate Total Bookings ───────────────────────
customers_df["computed_total"] = (
    customers_df["completed_rides"] +
    customers_df["cancelled_rides"] +
    customers_df["incomplete_rides"]
)
total_mismatch = customers_df[
    customers_df["total_bookings"] != customers_df["computed_total"]
]
print(f"Total bookings mismatches: {len(total_mismatch)}")
# Expected: 0 
customers_df.drop(columns=["computed_total"], inplace=True)

Total bookings mismatches: 0


#### 5.4.5 Remove Duplicates

In [14]:
# ── 5.4.5 Remove Duplicates ──────────────────────────────
print(f"Customer duplicates: {customers_df.duplicated().sum()}")
# Expected: 0 

Customer duplicates: 0


#### 5.4.6 Final Customers Check

In [15]:
# ── 5.2.6 Final Customers Check ─────────────────────────
print("\nCustomers nulls:\n", customers_df.isnull().sum().sum())
print("Customers shape:", customers_df.shape)
# Expected: 0 nulls, (10000, 13) 


Customers nulls:
 0
Customers shape: (10000, 13)


### 5.5 Drivers Cleaning

#### 5.5.1 Standardise Categoricals

In [16]:
# ════════════════════════════════════════════════════════
# 5.5 DRIVERS CLEANING
# ════════════════════════════════════════════════════════

# ── 5.5.1 Standardise Categoricals ──────────────────────
for col in ["driver_city", "vehicle_type"]:
    drivers_df[col] = drivers_df[col].str.strip().str.title()

#### 5.5.2 Validate Driver Age (18–65)

In [17]:
# ── 5.5.2 Validate Driver Age (18–65) ───────────────────
age_outliers = drivers_df[
    (drivers_df["driver_age"] < 18) |
    (drivers_df["driver_age"] > 65)
]
print(f"Driver age outliers: {len(age_outliers)}")
# Expected: 0 

Driver age outliers: 0


#### 5.5.3 Flag Experience vs Age Violations

In [18]:
# ── 5.5.3 Flag Experience vs Age Violations ─────────────
# Rule: driver cannot have worked before age 18
drivers_df["experience_outlier_flag"] = (
    drivers_df["driver_experience_years"] >
    (drivers_df["driver_age"] - 18)
).astype(int)
print(f"Experience outliers flagged: {drivers_df['experience_outlier_flag'].sum()}")
# 605 flagged 
# Capping will be done AFTER train/test split to avoid data leakage

Experience outliers flagged: 605


#### 5.5.4 Derive Rejected Rides

In [19]:
# ── 5.5.4 Derive Rejected Rides ─────────────────────────
# Correct formula discovered:
# total_assigned = accepted_rides + rejected_rides
# accepted_rides includes incomplete_rides as a subset
drivers_df["rejected_rides"] = (
    drivers_df["total_assigned_rides"] - drivers_df["accepted_rides"]
)
print(f"Negative rejected rides: {(drivers_df['rejected_rides'] < 0).sum()}")
# Expected: 0 
print(drivers_df["rejected_rides"].describe().round(2))

Negative rejected rides: 0
count    5000.00
mean        4.66
std         2.18
min         0.00
25%         3.00
50%         4.00
75%         6.00
max        15.00
Name: rejected_rides, dtype: float64


#### 5.5.5 Validate Acceptance Rate

In [20]:
# ── 5.5.5 Validate Acceptance Rate ──────────────────────
drivers_df["computed_accept_rate"] = (
    drivers_df["accepted_rides"] / drivers_df["total_assigned_rides"]
).round(2)
rate_mismatch = drivers_df[~np.isclose(
    drivers_df["acceptance_rate"],
    drivers_df["computed_accept_rate"], atol=0.01
)]
print(f"Acceptance rate mismatches: {len(rate_mismatch)}")
# Expected: 0 
drivers_df.drop(columns=["computed_accept_rate"], inplace=True)

Acceptance rate mismatches: 0


#### 5.5.6 Validate Delay Rate

In [21]:
# ── 5.5.6 Validate Delay Rate ───────────────────────────
drivers_df["computed_delay_rate"] = (
    drivers_df["delay_count"] / drivers_df["total_assigned_rides"]
).round(2)
delay_mismatch = drivers_df[~np.isclose(
    drivers_df["delay_rate"],
    drivers_df["computed_delay_rate"], atol=0.01
)]
print(f"Delay rate mismatches: {len(delay_mismatch)}")
# Expected: 0 
drivers_df.drop(columns=["computed_delay_rate"], inplace=True)

Delay rate mismatches: 0


#### 5.5.7 Remove Duplicates

In [22]:
# ── 5.5.7 Remove Duplicates ──────────────────────────────
print(f"Driver duplicates: {drivers_df.duplicated().sum()}")
# Expected: 0 

Driver duplicates: 0


#### 5.5.8 Final Drivers Check

In [23]:
# ── 5.5.8 Final Drivers Check ───────────────────────────
print("\nDrivers nulls:\n", drivers_df.isnull().sum().sum())
print("Drivers shape:", drivers_df.shape)
# Expected: 0 nulls, (5000, 16) 


Drivers nulls:
 0
Drivers shape: (5000, 16)


### 5.6 Location Demand Cleaning

#### 5.6.1 Standardise Categoricals

In [24]:
# ════════════════════════════════════════════════════════
# 5.6 LOCATION DEMAND CLEANING
# ════════════════════════════════════════════════════════

# ── 5.6.1 Standardise Categoricals ──────────────────────
for col in ["city", "vehicle_type", "demand_level"]:
    location_demand_df[col] = location_demand_df[col].str.strip().str.title()


#### 5.6.2 Validate Ride Counts

In [25]:
# ── 5.6.2 Validate Ride Counts ──────────────────────────
location_demand_df["ride_sum"] = (
    location_demand_df["completed_rides"] +
    location_demand_df["cancelled_rides"]
)
over_count = location_demand_df[
    location_demand_df["ride_sum"] > location_demand_df["total_requests"]
]
print(f"Rides exceeding total requests: {len(over_count)}")
# Expected: 0 
location_demand_df.drop(columns=["ride_sum"], inplace=True)

Rides exceeding total requests: 0


#### 5.6.3 Remove Duplicates

In [26]:
# ── 5.6.3 Remove Duplicates ──────────────────────────────
print(f"Location demand duplicates: {location_demand_df.duplicated().sum()}")
# Expected: 0 

Location demand duplicates: 0


#### 5.6.4 Final Location Demand Check

In [27]:
# ── 5.6.4 Final Location Demand Check ───────────────────
print("\nLocation Demand nulls:\n", location_demand_df.isnull().sum().sum())
print("Location Demand shape:", location_demand_df.shape)
# Expected: 0 nulls, (17941, 10) 



Location Demand nulls:
 0
Location Demand shape: (17941, 10)


### 5.7 Time Features Cleaning

#### 5.7.1 Fix Datetime Type

In [28]:
# ════════════════════════════════════════════════════════
# 5.7 TIME FEATURES CLEANING
# ════════════════════════════════════════════════════════

# ── 5.7.1 Fix Datetime Type ─────────────────────────────
time_features_df["datetime"] = pd.to_datetime(time_features_df["datetime"])
print("datetime dtype:", time_features_df["datetime"].dtype)
# Expected: datetime64[ns] 

datetime dtype: datetime64[ns]


#### 5.7.2 Standardise Categoricals

In [29]:
# ── 5.7.2 Standardise Categoricals ──────────────────────
for col in ["day_of_week", "season"]:
    time_features_df[col] = time_features_df[col].str.strip().str.title()

#### 5.7.3 Flag is_holiday = All Zero Issue

In [30]:
# ── 5.7.3 Flag is_holiday = All Zero Issue ───────────────
print(f"is_holiday unique values: {time_features_df['is_holiday'].unique()}")
print(f"is_holiday sum: {time_features_df['is_holiday'].sum()}")
# ⚠️ WARNING: is_holiday is all 0
# Verify if holiday data was loaded correctly in source

is_holiday unique values: [0]
is_holiday sum: 0


#### 5.7.4 Remove Duplicates

In [31]:
# ── 5.7.4 Remove Duplicates ──────────────────────────────
print(f"Time features duplicates: {time_features_df.duplicated().sum()}")
# Expected: 0 

Time features duplicates: 0


#### 5.7.5 Final Time Features Check

In [32]:
# ── 5.7.5 Final Time Features Check ─────────────────────
print("\nTime Features nulls:\n", time_features_df.isnull().sum().sum())
print("Time Features shape:", time_features_df.shape)
# Expected: 0 nulls, (8760, 7) 


Time Features nulls:
 0
Time Features shape: (8760, 7)


### 5.8 FINAL SUMMARY CHECK — ALL DATASETS

In [33]:
# ════════════════════════════════════════════════════════
# 5.8 FINAL SUMMARY CHECK — ALL DATASETS
# ════════════════════════════════════════════════════════
datasets = {
    "Bookings":        bookings_df,
    "Customers":       customers_df,
    "Drivers":         drivers_df,
    "Location Demand": location_demand_df,
    "Time Features":   time_features_df
}

print("\n" + "="*55)
print("FINAL DATA CLEANING SUMMARY")
print("="*55)
for name, df in datasets.items():
    nulls     = df.isnull().sum().sum()
    dupes     = df.duplicated().sum()
    print(f"{name:20s} | Shape: {str(df.shape):15s} | Nulls: {nulls:6} | Dupes: {dupes}")

# Expected output:
# Bookings             | Shape: (100000, 25)   | Nulls:  31654 | Dupes: 0  ← intentional NaN
# Customers            | Shape: (10000, 13)    | Nulls:      0 | Dupes: 0
# Drivers              | Shape: (5000, 16)     | Nulls:      0 | Dupes: 0
# Location Demand      | Shape: (17941, 10)    | Nulls:      0 | Dupes: 0
# Time Features        | Shape: (8760, 7)      | Nulls:      0 | Dupes: 0


FINAL DATA CLEANING SUMMARY
Bookings             | Shape: (100000, 23)    | Nulls:  31654 | Dupes: 0
Customers            | Shape: (10000, 13)     | Nulls:      0 | Dupes: 0
Drivers              | Shape: (5000, 16)      | Nulls:      0 | Dupes: 0
Location Demand      | Shape: (17941, 10)     | Nulls:      0 | Dupes: 0
Time Features        | Shape: (8760, 7)       | Nulls:      0 | Dupes: 0


### 5.9 Saved Cleaned Datasets

In [34]:
# ════════════════════════════════════════════════════════
# 5.9 SAVE CLEANED DATASETS
# ════════════════════════════════════════════════════════
bookings_df.to_csv("../data/cleaned/bookings_cleaned.csv",               index=False)
customers_df.to_csv("../data/cleaned/customers_cleaned.csv",             index=False)
drivers_df.to_csv("../data/cleaned/drivers_cleaned.csv",                 index=False)
location_demand_df.to_csv("../data/cleaned/location_demand_cleaned.csv", index=False)
time_features_df.to_csv("../data/cleaned/time_features_cleaned.csv",     index=False)

print("\nAll datasets cleaned and saved successfully!")


All datasets cleaned and saved successfully!
